# NTU KTP — DQ Engine Evaluation Pipeline (Aizle Consumer Banking)

**What this notebook does:**
1. Load 4 clean synthetic banking datasets (Aizle Consumer Banking package)
2. Inject controlled errors at 1%, 5%, 10% noise levels with full ground truth
3. Upload rules JSON + noisy CSVs to S3 → pipeline fires automatically
4. Wait for all 12 Step Functions executions to complete
5. Download results and compute Precision / Recall / F1
6. Generate paper-ready figures

---
### Before you run
Add the following to **Colab Secrets** (key icon in the left sidebar):
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`

Upload the following parquet files from `aizle-consumer_banking-small-9_1_0/data/` when prompted:
- `consumer_banking-small-en_GB-v9_1_0-bank1_personal_transactions.parquet`
- `consumer_banking-small-en_GB-v9_1_0-bank1_personal_customers.parquet`
- `consumer_banking-small-en_GB-v9_1_0-bank1_personal_loan_accounts.parquet`
- `consumer_banking-small-en_GB-v9_1_0-bank1_personal_credit_card_accounts.parquet`

Upload the 4 rules JSON files from `rules/aizle/` in the repo:
- `transactions_rules.json`
- `customers_rules.json`
- `loan_accounts_rules.json`
- `credit_card_accounts_rules.json`

## 0. Install & Import

In [ ]:
!pip install pyarrow boto3 pandas numpy matplotlib seaborn --quiet

In [ ]:
import os, io, json, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from datetime import datetime, timedelta
from collections import defaultdict

try:
    from google.colab import userdata, files
    IN_COLAB = True
    def get_secret(k):
        try:
            return userdata.get(k)
        except Exception:
            return None
except ImportError:
    IN_COLAB = False
    def get_secret(k): return os.environ.get(k, '')

print('Environment:', 'Google Colab' if IN_COLAB else 'Local Jupyter')

## 1. Configuration

In [ ]:
# ── AWS ───────────────────────────────────────────────────────────────────
AWS_REGION        = 'us-east-1'
S3_INBOX_BUCKET   = 'dq-investigator-inbox-dev'
S3_OUTPUT_BUCKET  = 'dq-investigator-output-dev'
S3_INBOX_PREFIX   = 'incoming/'
S3_RULES_PREFIX   = 'rules/aizle/'
STATE_MACHINE_ARN = 'arn:aws:states:us-east-1:230802932710:stateMachine:dq-pipeline-dev'

# ── Noise levels ─────────────────────────────────────────────────────────
NOISE_LEVELS = [0.01, 0.05, 0.10]
RANDOM_SEED  = 42

# ── Dataset definitions ───────────────────────────────────────────────────
# Maps notebook dataset name → parquet filename stem & rules filename
DATASET_CONFIG = {
    'transactions':        'consumer_banking-small-en_GB-v9_1_0-bank1_personal_transactions',
    'customers':           'consumer_banking-small-en_GB-v9_1_0-bank1_personal_customers',
    'loan_accounts':       'consumer_banking-small-en_GB-v9_1_0-bank1_personal_loan_accounts',
    'credit_card_accounts':'consumer_banking-small-en_GB-v9_1_0-bank1_personal_credit_card_accounts',
}

RULES_FILES = {
    'transactions':         'transactions_rules.json',
    'customers':            'customers_rules.json',
    'loan_accounts':        'loan_accounts_rules.json',
    'credit_card_accounts': 'credit_card_accounts_rules.json',
}

# ── AWS session ───────────────────────────────────────────────────────────
boto_session = boto3.Session(
    aws_access_key_id     = get_secret('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = get_secret('AWS_SECRET_ACCESS_KEY'),
    region_name           = AWS_REGION,
)
s3_client  = boto_session.client('s3')
sfn_client = boto_session.client('stepfunctions')

print('AWS session ready.')

## 2. Upload Rules JSON Files to S3
Do this once — the pipeline Lambda auto-detects them by convention.

In [ ]:
rules_data = {}   # ds_name → parsed rules dict

if IN_COLAB:
    print('Upload all 4 rules JSON files from rules/aizle/ in the repo')
    print('(transactions_rules.json, customers_rules.json, loan_accounts_rules.json, credit_card_accounts_rules.json)\n')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        rules = json.loads(data.decode('utf-8'))
        # Derive dataset name from filename
        ds_name = fname.replace('_rules.json', '')
        rules_data[ds_name] = rules
        # Upload to S3
        s3_key = f'{S3_RULES_PREFIX}{fname}'
        s3_client.put_object(
            Bucket=S3_INBOX_BUCKET,
            Key=s3_key,
            Body=data,
            ContentType='application/json'
        )
        print(f'  Uploaded s3://{S3_INBOX_BUCKET}/{s3_key}  ({len(rules)} column rules)')
else:
    # Local: load from repo
    for ds_name, fname in RULES_FILES.items():
        path = f'../rules/aizle/{fname}'
        with open(path) as f:
            rules = json.load(f)
        rules_data[ds_name] = rules
        s3_key = f'{S3_RULES_PREFIX}{fname}'
        with open(path, 'rb') as f:
            s3_client.put_object(Bucket=S3_INBOX_BUCKET, Key=s3_key,
                                 Body=f.read(), ContentType='application/json')
        print(f'  Uploaded s3://{S3_INBOX_BUCKET}/{s3_key}')

print(f'\nRules uploaded for: {list(rules_data.keys())}')

## 3. Load Aizle Parquet Files

In [ ]:
datasets_clean = {}   # ds_name → clean DataFrame

if IN_COLAB:
    print('Upload all 4 parquet files from aizle-consumer_banking-small-9_1_0/data/\n')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        buf = io.BytesIO(data)
        df  = pd.read_parquet(buf)
        # Match to dataset name
        for ds_name, stem in DATASET_CONFIG.items():
            if stem in fname or ds_name in fname:
                datasets_clean[ds_name] = df.reset_index(drop=True)
                print(f'  {ds_name}: {df.shape[0]} rows x {df.shape[1]} cols')
                break
else:
    base = os.path.expanduser('~/Downloads/aizle-consumer_banking-small-9_1_0/data')
    for ds_name, stem in DATASET_CONFIG.items():
        path = os.path.join(base, f'{stem}.parquet')
        df = pd.read_parquet(path)
        datasets_clean[ds_name] = df.reset_index(drop=True)
        print(f'  {ds_name}: {df.shape[0]} rows x {df.shape[1]} cols')

print(f'\nAll {len(datasets_clean)} datasets loaded.')

## 4. Error Injection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# inject_errors — Aizle-aware controlled error injection
# ═══════════════════════════════════════════════════════════════════════════

def _mangle_sort_code(val):
    """Turn '01-01-01' into an invalid sort code."""
    variants = ['010101', '01/01/01', '01 01 01', 'XX-YY-ZZ', '99-99-99']
    return np.random.default_rng().choice(variants)

def _mangle_postcode(val):
    """Turn 'SW1A 2AA' into an invalid postcode."""
    if not isinstance(val, str): return 'INVALID'
    variants = [
        val.lower(),                        # lowercase
        val.replace(' ', ''),               # no space
        val[:2] + '99' + val[4:],           # wrong district
        'ZZ99 9ZZ',                         # clearly invalid
    ]
    return str(np.random.default_rng().choice(variants))

def _mangle_phone(val):
    """Turn '+447123456789' into an invalid phone."""
    if not isinstance(val, str): return '0000'
    variants = [
        val.replace('+44', '0'),            # wrong prefix
        val[1:],                            # strip +
        val + '0',                          # too long
        '123456',                           # too short
    ]
    return str(np.random.default_rng().choice(variants))

def inject_errors(df_clean, ds_name, noise_level=0.05, seed=42):
    """
    Inject controlled errors into a clean Aizle dataframe.

    Returns
    -------
    df_noisy     : DataFrame with injected errors
    ground_truth : DataFrame [row_id | column | issue_type | original_value | injected_value]
    """
    rng = np.random.default_rng(seed)
    df  = df_clean.copy().reset_index(drop=True)
    gt  = []
    used = set()

    n_rows   = len(df)
    n_errors = max(2, int(n_rows * noise_level))

    # ── Column type classification ────────────────────────────────────────
    sort_code_cols  = [c for c in df.columns if 'sort_code' in c.lower()]
    postcode_cols   = [c for c in df.columns if 'postcode' in c.lower()]
    phone_cols      = [c for c in df.columns if any(k in c.lower() for k in ['mobile', 'land_line', 'phone'])]
    date_cols       = [c for c in df.columns
                       if any(k in c.lower() for k in ['date', 'dob', '_at', 'timestamp'])
                       and df[c].dtype in ['object', 'datetime64[ns]', 'datetime64[ns, UTC]']
                         or pd.api.types.is_datetime64_any_dtype(df[c])]
    numeric_cols    = [c for c in df.columns
                       if df[c].dtype in [np.float64, np.int64, float, int]
                       and c not in date_cols]
    categorical_cols= [c for c in df.columns
                       if df[c].dtype == object
                       and df[c].nunique() <= 25
                       and c not in sort_code_cols + postcode_cols + phone_cols]
    all_cols        = [c for c in df.columns
                       if c not in ['tid', 'account_id', 'customer_id', 'agent_id',
                                    'account_number', 'counterparty_account_number',
                                    'narrative', 'transaction_category']]

    # ── Pre-build shuffled queues of valid row indices per column ─────────
    # Replaces the original O(n) list-comprehension inside pick() that caused
    # 13+ minute hangs on the transactions dataset (~25K rows).
    # Each column's valid indices are computed once with vectorised ops and
    # shuffled; pick() walks forward through the queue in O(1) amortized.
    _col_queue: dict = {}
    _col_ptr:   dict = {}
    for _c in df.columns:
        _notna  = df[_c].notna().values
        _svals  = df[_c].astype(str).values
        _bad    = np.isin(_svals, ['', 'nan', 'None', 'NaT', 'nat', '<NA>'])
        _valid  = np.where(_notna & ~_bad)[0].copy()
        rng.shuffle(_valid)
        _col_queue[_c] = _valid
        _col_ptr[_c]   = 0

    def pick(col):
        q   = _col_queue.get(col, np.array([], dtype=int))
        ptr = _col_ptr.get(col, 0)
        while ptr < len(q):
            i    = int(q[ptr])
            ptr += 1
            if (i, col) not in used:
                _col_ptr[col] = ptr
                return i
        _col_ptr[col] = ptr   # column exhausted
        return None

    def record(row, col, itype, orig, injected):
        gt.append({'row_id': row, 'column': col, 'issue_type': itype,
                   'original_value': str(orig), 'injected_value': str(injected)})
        used.add((row, col))

    per = max(1, n_errors // 6)   # errors split across 6 types

    # ── 1. missing_value ─────────────────────────────────────────────────
    for _ in range(per):
        col = str(rng.choice(all_cols))
        row = pick(col)
        if row is None: continue
        orig = df.at[row, col]
        inj  = rng.choice([None, '', 'N/A', 'NULL'])
        df.at[row, col] = inj
        record(row, col, 'missing_value', orig, inj)

    # ── 2. format_error — sort code ───────────────────────────────────────
    if sort_code_cols:
        for _ in range(per):
            col = str(rng.choice(sort_code_cols))
            row = pick(col)
            if row is None: continue
            orig = df.at[row, col]
            inj  = _mangle_sort_code(orig)
            df.at[row, col] = inj
            record(row, col, 'format_error', orig, inj)

    # ── 3. format_error — postcode ────────────────────────────────────────
    if postcode_cols:
        for _ in range(per):
            col = str(rng.choice(postcode_cols))
            row = pick(col)
            if row is None: continue
            orig = df.at[row, col]
            inj  = _mangle_postcode(str(orig))
            df.at[row, col] = inj
            record(row, col, 'format_error', orig, inj)

    # ── 4. format_error — phone ───────────────────────────────────────────
    if phone_cols:
        for _ in range(per):
            col = str(rng.choice(phone_cols))
            row = pick(col)
            if row is None: continue
            orig = df.at[row, col]
            inj  = _mangle_phone(str(orig))
            df.at[row, col] = inj
            record(row, col, 'format_error', orig, inj)

    # ── 5. range_error — numeric ──────────────────────────────────────────
    if numeric_cols:
        for _ in range(per * 2):   # give more weight to numeric
            col = str(rng.choice(numeric_cols))
            row = pick(col)
            if row is None: continue
            try:
                orig_f = float(df.at[row, col])
            except Exception:
                continue
            inj = round(float(rng.choice([
                rng.uniform(-9999999, -1000),
                rng.uniform(1e7, 1e9),
            ])), 2)
            df.at[row, col] = inj
            record(row, col, 'range_error', orig_f, inj)

    # ── 6. range_error — date (past or far future) ────────────────────────
    if date_cols:
        for _ in range(per):
            col = str(rng.choice(date_cols))
            row = pick(col)
            if row is None: continue
            orig = df.at[row, col]
            if rng.random() < 0.5:
                inj = f'{int(rng.integers(1800, 1900))}-{int(rng.integers(1,13)):02d}-{int(rng.integers(1,28)):02d}'
            else:
                inj = (datetime.now() + timedelta(days=int(rng.integers(3650, 7300)))).strftime('%Y-%m-%d')
            df.at[row, col] = inj
            record(row, col, 'range_error', str(orig), inj)

    # ── 7. corpus_mismatch — categorical ──────────────────────────────────
    if categorical_cols:
        for _ in range(per):
            col = str(rng.choice(categorical_cols))
            row = pick(col)
            if row is None: continue
            orig = str(df.at[row, col])
            # Insert a random char to corrupt the value
            pos  = int(rng.integers(1, max(2, len(orig))))
            char = str(rng.choice(list('abcdefghijklmnopqrstuvwxyz123')))
            inj  = orig[:pos] + char + orig[pos:]
            df.at[row, col] = inj
            record(row, col, 'corpus_mismatch', orig, inj)

    # ── 8. duplicate rows ─────────────────────────────────────────────────
    n_dupes = max(1, per // 2)
    dup_idx = rng.choice(n_rows, size=n_dupes, replace=False)
    for orig_idx in dup_idx:
        gt.append({'row_id': len(df), 'column': 'ALL',
                   'issue_type': 'duplicate',
                   'original_value': int(orig_idx),
                   'injected_value': f'duplicate_of_{int(orig_idx)}'})
    df = pd.concat([df, df.iloc[dup_idx].copy()], ignore_index=True)

    gt_df  = pd.DataFrame(gt)
    counts = gt_df['issue_type'].value_counts().to_dict()
    print(f'  {ds_name} {noise_level*100:.0f}%: injected {len(gt_df)} errors '
          f'into {len(df)} rows  {counts}')
    return df, gt_df

print('inject_errors ready.')

In [ ]:
# Run injection across all 4 datasets × 3 noise levels
# injected[ds_name][noise_pct] = (df_noisy, df_ground_truth)
injected = defaultdict(dict)

print('Running error injection...\n')
for ds_name, df_clean in datasets_clean.items():
    print(f'Dataset: {ds_name}  ({len(df_clean)} rows x {len(df_clean.columns)} cols)')
    for noise in NOISE_LEVELS:
        pct = int(noise * 100)
        df_noisy, df_gt = inject_errors(df_clean, ds_name,
                                         noise_level=noise,
                                         seed=RANDOM_SEED + pct)
        injected[ds_name][pct] = (df_noisy, df_gt)
    print()

print('All injections complete.')

## 5. Upload Noisy CSVs to S3 & Trigger Pipeline

In [ ]:
def upload_csv(df, bucket, key):
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    s3_client.put_object(
        Bucket=bucket, Key=key,
        Body=buf.getvalue().encode('utf-8'),
        ContentType='text/csv'
    )
    print(f'  Uploaded s3://{bucket}/{key}  ({len(df)} rows)')


print('Uploading noisy datasets to S3 inbox...\n')
print('NOTE: The Lambda will auto-detect rules/aizle/{dataset}_rules.json')
print('      and pass it to the DQ engine automatically.\n')

for ds_name in datasets_clean:
    for noise in NOISE_LEVELS:
        pct = int(noise * 100)
        df_noisy, _ = injected[ds_name][pct]
        key = f'{S3_INBOX_PREFIX}{ds_name}_{pct:02d}pct.csv'
        upload_csv(df_noisy, S3_INBOX_BUCKET, key)
        time.sleep(2)   # small gap so EventBridge registers each event

print('\nAll uploads complete.')
print('EventBridge → Lambda → Step Functions executions starting automatically...')
print('Waiting 20s for executions to register...')
time.sleep(20)

## 6. Wait for All Executions to Complete

In [ ]:
def list_executions(n=50):
    resp = sfn_client.list_executions(
        stateMachineArn=STATE_MACHINE_ARN,
        maxResults=n
    )
    return resp.get('executions', [])


def wait_for_pipeline(timeout_min=40):
    deadline = time.time() + timeout_min * 60
    print(f'Polling Step Functions (timeout: {timeout_min} min)...\n')
    while time.time() < deadline:
        execs     = list_executions()
        running   = [e for e in execs if e['status'] == 'RUNNING']
        succeeded = [e for e in execs if e['status'] == 'SUCCEEDED']
        failed    = [e for e in execs if e['status'] == 'FAILED']
        ts        = datetime.now().strftime('%H:%M:%S')
        print(f'  [{ts}]  Running: {len(running)}   Succeeded: {len(succeeded)}   Failed: {len(failed)}')
        if len(running) == 0:
            print('\nAll executions finished.')
            return execs
        time.sleep(30)
    print('Timeout reached — proceeding with available results.')
    return list_executions()


all_execs = wait_for_pipeline()
succeeded = [e for e in all_execs if e['status'] == 'SUCCEEDED']
failed    = [e for e in all_execs if e['status'] == 'FAILED']
print(f'\nFinal: {len(succeeded)} succeeded, {len(failed)} failed out of {len(all_execs)} total')

## 7. Download Results from S3

In [ ]:
def list_s3(bucket, prefix):
    paginator = s3_client.get_paginator('list_objects_v2')
    keys = []
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        keys += [o['Key'] for o in page.get('Contents', [])]
    return keys

def read_csv_s3(bucket, key):
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

def read_json_s3(bucket, key):
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(obj['Body'].read().decode('utf-8'))


# results[ds_name][pct] = {'report': {...}, 'issues': df}
results = defaultdict(dict)

all_keys = list_s3(S3_OUTPUT_BUCKET, 'reports/')
print(f'Found {len(all_keys)} files in output bucket.\n')

for ds_name in datasets_clean:
    for noise in NOISE_LEVELS:
        pct       = int(noise * 100)
        stem      = f'{ds_name}_{pct:02d}pct'
        rpt_key   = next((k for k in all_keys if stem in k and k.endswith('_report.json')), None)
        iss_key   = next((k for k in all_keys if stem in k and k.endswith('_issues.csv')), None)

        if rpt_key:
            report = read_json_s3(S3_OUTPUT_BUCKET, rpt_key)
            results[ds_name][pct] = {'report': report}
            print(f'  {ds_name} {pct}%  score={report.get("overall_score")}  '
                  f'issues={report.get("issues_count")}')
        else:
            print(f'  MISSING report for {ds_name} {pct}%')

        if iss_key and ds_name in results and pct in results[ds_name]:
            results[ds_name][pct]['issues'] = read_csv_s3(S3_OUTPUT_BUCKET, iss_key)

print('\nDownload complete.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DIAGNOSTICS — run after Section 7 to investigate:
#   A. Why transactions results are missing
#   B. What the engine is actually flagging (correct issue labels)
#   C. Why customers has 500+ false positives
#   D. Whether rules files exist in S3 inbox
# ═══════════════════════════════════════════════════════════════════════════

SEP = '─' * 70

# ── A. S3 output bucket — full key listing ────────────────────────────────
print('A. ALL KEYS IN OUTPUT BUCKET')
print(SEP)
for k in sorted(all_keys):
    print(f'  {k}')
print(f'\nTotal: {len(all_keys)} files')
print()

# Specifically check for transactions
txn_keys = [k for k in all_keys if 'transaction' in k.lower()]
print(f'  Keys containing "transaction": {txn_keys if txn_keys else "NONE — pipeline did not produce output for transactions"}')
print()

# ── B. Step Functions — check execution status for each dataset ───────────
print('B. STEP FUNCTIONS EXECUTION STATUS (last 50)')
print(SEP)
try:
    all_execs_diag = sfn_client.list_executions(
        stateMachineArn=STATE_MACHINE_ARN, maxResults=50
    )['executions']
    for e in all_execs_diag:
        name   = e.get('name', '')
        status = e.get('status', '')
        start  = e.get('startDate', '').strftime('%H:%M:%S') if hasattr(e.get('startDate', ''), 'strftime') else str(e.get('startDate', ''))
        print(f'  [{status:12s}]  {start}  {name}')
except Exception as ex:
    print(f'  Could not list executions: {ex}')
print()

# ── C. Rules files — confirm they exist in inbox bucket ──────────────────
print('C. RULES FILES IN INBOX BUCKET')
print(SEP)
for ds_name, fname in RULES_FILES.items():
    key = f'{S3_RULES_PREFIX}{fname}'
    try:
        meta = s3_client.head_object(Bucket=S3_INBOX_BUCKET, Key=key)
        size = meta['ContentLength']
        print(f'  [OK]      s3://{S3_INBOX_BUCKET}/{key}  ({size} bytes)')
    except Exception:
        print(f'  [MISSING] s3://{S3_INBOX_BUCKET}/{key}')
print()

# ── D. Actual issue labels in _issues.csv files ───────────────────────────
print('D. ACTUAL ENGINE ISSUE LABELS IN _issues.csv FILES')
print(SEP)
print('(These must match ENGINE_TO_CANON in Section 8)\n')

all_labels = {}
for ds_name in list(results.keys()):
    for pct in list(results[ds_name].keys()):
        iss_df = results[ds_name][pct].get('issues')
        if iss_df is None or iss_df.empty:
            continue
        if 'issue' not in iss_df.columns:
            print(f'  {ds_name} {pct}%: "issue" column NOT FOUND — columns are: {list(iss_df.columns)}')
            continue
        counts = iss_df['issue'].value_counts()
        for label, cnt in counts.items():
            all_labels[label] = all_labels.get(label, 0) + cnt
        # Show just one dataset at each noise level for brevity
        if pct == 5:
            print(f'  {ds_name} {pct}% — top issue labels:')
            for label, cnt in counts.head(10).items():
                mapped = {
                    'missing': 'missing_value', 'missing_value': 'missing_value',
                    'format_error': 'format_error', 'pattern': 'format_error',
                    'regex mismatch': 'format_error', 'not number': 'format_error',
                    'invalid date': 'format_error',
                    '<min': 'range_error', '>max': 'range_error',
                    'range_error': 'range_error', 'numeric_outlier': 'range_error',
                    'logical_error': 'range_error',
                    'corpus_mismatch': 'corpus_mismatch',
                    'standardisation_error': 'corpus_mismatch',
                    'duplicate': 'duplicate', 'duplicate_value': 'duplicate',
                    'near_duplicate': 'duplicate',
                    'rare_category': 'discovery', 'text_length_outlier': 'discovery',
                    'ml_anomaly': 'discovery',
                }.get(str(label).lower(), 'UNMAPPED ← ADD TO ENGINE_TO_CANON')
                print(f'    {cnt:6d}  "{label}"  →  {mapped}')
            print()

print('ALL UNIQUE LABELS ACROSS ALL FILES:')
for label, cnt in sorted(all_labels.items(), key=lambda x: -x[1]):
    print(f'  {cnt:6d}  "{label}"')
print()

# ── E. Customers FP deep-dive ─────────────────────────────────────────────
print('E. CUSTOMERS 1% — WHY 587 FALSE POSITIVES?')
print(SEP)
if 'customers' in results and 1 in results['customers']:
    iss = results['customers'][1].get('issues')
    if iss is not None and not iss.empty:
        print(f'  Total issues reported: {len(iss)}')
        print(f'  Columns: {list(iss.columns)}\n')

        if 'column' in iss.columns and 'issue' in iss.columns:
            print('  Top (column, issue) pairs:')
            combo = iss.groupby(['column', 'issue']).size().sort_values(ascending=False)
            for (col, iss_label), cnt in combo.head(20).items():
                print(f'    {cnt:6d}  column="{col}"  issue="{iss_label}"')
            print()

            print('  Sample rows for the top offending column/issue:')
            top_col, top_iss = combo.index[0]
            sample = iss[(iss['column'] == top_col) & (iss['issue'] == top_iss)].head(5)
            print(sample.to_string(index=False))
        else:
            print('  Raw columns found:', list(iss.columns))
            print(iss.head(10).to_string())
    else:
        print('  No issues dataframe for customers 1%')
else:
    print('  customers 1% not in results dict')

print()
print('Diagnostics complete.')
print('Use the output above to:')
print('  1. Check Step Functions for failed transactions executions')
print('  2. Add any UNMAPPED labels to ENGINE_TO_CANON in Section 8')
print('  3. Understand which column/rule is causing the customers FP explosion')

## 7b. Diagnostics — Root Cause Analysis
Run this cell after Section 7 to understand missing results and false positive explosion before proceeding to F1 computation.

## 8. Compute Precision / Recall / F1

In [ ]:
# ── Normalisation maps ────────────────────────────────────────────────────
ENGINE_TO_CANON = {
    'missing':               'missing_value',
    'missing_value':         'missing_value',
    'format_error':          'format_error',
    'pattern':               'format_error',
    'regex mismatch':        'format_error',
    'not number':            'format_error',
    'invalid date':          'format_error',
    '<min':                  'range_error',
    '>max':                  'range_error',
    'range_error':           'range_error',
    'numeric_outlier':       'range_error',
    'logical_error':         'range_error',
    'corpus_mismatch':       'corpus_mismatch',
    'standardisation_error': 'corpus_mismatch',
    'duplicate':             'duplicate',
    'duplicate_value':       'duplicate',
    'near_duplicate':        'duplicate',
    'rare_category':         'discovery',
    'text_length_outlier':   'discovery',
    'ml_anomaly':            'discovery',
}
EXCLUDED = {'discovery', 'other'}


def compute_f1(issues_df, gt_df):
    """
    Match (row_id, column, issue_type) tuples.
    GT uses canonical types already (from inject_errors).
    Engine output is normalised via ENGINE_TO_CANON.
    Row-level entries (column=ALL) are excluded from cell matching.
    """
    if issues_df is None or issues_df.empty:
        return {'precision': 0, 'recall': 0, 'f1': 0,
                'tp': 0, 'fp': 0, 'fn': 0, 'gt_total': 0}, {}

    # Ground truth — cell-level only (exclude duplicate row entries)
    gt = gt_df[~gt_df['column'].isin(['ALL', 'all_columns'])].copy()
    gt_set = set(zip(gt['row_id'].astype(int),
                     gt['column'].astype(str),
                     gt['issue_type'].astype(str)))

    # Detections — normalise issue type
    det = issues_df.copy()
    det['canon'] = det['issue'].map(ENGINE_TO_CANON).fillna('other')
    det = det[~det['canon'].isin(EXCLUDED)]
    det_set = set(zip(det['row_id'].astype(int),
                      det['column'].astype(str),
                      det['canon'].astype(str)))

    tp = len(det_set & gt_set)
    fp = len(det_set - gt_set)
    fn = len(gt_set - det_set)

    p  = tp / (tp + fp) if (tp + fp) > 0 else 0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

    overall = {'precision': round(p, 3), 'recall': round(r, 3),
               'f1': round(f1, 3), 'tp': tp, 'fp': fp, 'fn': fn,
               'gt_total': len(gt_set)}

    # Per issue type
    per_type = {}
    all_types = (set(gt['issue_type'].unique()) |
                 set(det['canon'].unique())) - EXCLUDED
    for itype in all_types:
        dt = {(r, c) for r, c, t in det_set if t == itype}
        gt_t = {(r, c) for r, c, t in gt_set if t == itype}
        tp_t = len(dt & gt_t)
        fp_t = len(dt - gt_t)
        fn_t = len(gt_t - dt)
        p_t  = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
        r_t  = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
        f_t  = 2*p_t*r_t/(p_t+r_t) if (p_t+r_t) > 0 else 0
        per_type[itype] = {'precision': round(p_t,3), 'recall': round(r_t,3),
                           'f1': round(f_t,3), 'gt_count': len(gt_t)}

    return overall, per_type


# ── Compute metrics ───────────────────────────────────────────────────────
metrics_table = []

print(f'{"Dataset":30s} {"Noise":>6s}  {"P":>6s}  {"R":>6s}  {"F1":>6s}  {"GT":>5s}  {"TP":>5s}  {"FP":>5s}')
print('-' * 75)

for ds_name in datasets_clean:
    for noise in NOISE_LEVELS:
        pct = int(noise * 100)
        if ds_name not in results or pct not in results[ds_name]:
            print(f'  MISSING: {ds_name} {pct}%')
            continue

        run       = results[ds_name][pct]
        issues_df = run.get('issues')
        gt_df     = injected[ds_name][pct][1]
        overall, per_type = compute_f1(issues_df, gt_df)

        print(f'{ds_name:30s} {pct:5d}%  '
              f'{overall["precision"]:6.3f}  {overall["recall"]:6.3f}  '
              f'{overall["f1"]:6.3f}  {overall["gt_total"]:5d}  '
              f'{overall["tp"]:5d}  {overall["fp"]:5d}')

        metrics_table.append({
            'dataset': ds_name, 'noise_pct': pct,
            'overall_score': run['report'].get('overall_score', 0),
            **{f'overall_{k}': v for k, v in overall.items()},
            'per_type': per_type,
        })

df_metrics = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_type'}
                            for r in metrics_table])

## 9. Paper-Ready Results Table

In [ ]:
display_cols = ['dataset', 'noise_pct', 'overall_precision', 'overall_recall',
                'overall_f1', 'overall_tp', 'overall_fp', 'overall_fn',
                'overall_gt_total', 'overall_score']
df_display = df_metrics[display_cols].copy()
df_display.columns = ['Dataset', 'Noise%', 'Precision', 'Recall', 'F1',
                      'TP', 'FP', 'FN', 'GT Total', 'Quality Score']
print('=== EVALUATION RESULTS ===')
print(df_display.to_string(index=False))

## 10. Figures

In [ ]:
COLOURS = {
    'transactions':         '#00e5ff',
    'customers':            '#00ff41',
    'loan_accounts':        '#ffd700',
    'credit_card_accounts': '#ff6b35',
}
LABELS = {
    'transactions':         'Transactions',
    'customers':            'Customers',
    'loan_accounts':        'Loan Accounts',
    'credit_card_accounts': 'Credit Card Accounts',
}
NOISE_TICKS = [int(n*100) for n in NOISE_LEVELS]


# ── Figure 1: F1 vs Noise Level ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('#0a0a0a'); fig.patch.set_facecolor('#0d1117')

for ds in datasets_clean:
    d = df_metrics[df_metrics['dataset'] == ds].sort_values('noise_pct')
    if d.empty: continue
    ax.plot(d['noise_pct'], d['overall_f1'],
            marker='o', linewidth=2.5, markersize=8,
            color=COLOURS[ds], label=LABELS[ds])

ax.set_xlabel('Noise Level (%)', color='white', fontsize=12)
ax.set_ylabel('F1 Score', color='white', fontsize=12)
ax.set_title('F1 Score vs Noise Level — Aizle Consumer Banking', color='white', fontsize=13, pad=15)
ax.set_ylim(0, 1.05)
ax.set_xticks(NOISE_TICKS)
ax.set_xticklabels([f'{n}%' for n in NOISE_TICKS], color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#333')
ax.grid(alpha=0.2, color='#444')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
plt.tight_layout()
plt.savefig('fig1_f1_vs_noise.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_f1_vs_noise.png')

In [ ]:
# ── Figure 2: P/R/F1 per Issue Type at 5% noise (avg across datasets) ─────
per_type_avg = defaultdict(lambda: {'precision': [], 'recall': [], 'f1': []})
for row in metrics_table:
    if row['noise_pct'] != 5: continue
    for itype, m in row['per_type'].items():
        per_type_avg[itype]['precision'].append(m['precision'])
        per_type_avg[itype]['recall'].append(m['recall'])
        per_type_avg[itype]['f1'].append(m['f1'])

itypes  = sorted(per_type_avg.keys())
p_vals  = [np.mean(per_type_avg[t]['precision']) for t in itypes]
r_vals  = [np.mean(per_type_avg[t]['recall'])    for t in itypes]
f_vals  = [np.mean(per_type_avg[t]['f1'])        for t in itypes]

x, w = np.arange(len(itypes)), 0.28
fig, ax = plt.subplots(figsize=(11, 5))
ax.set_facecolor('#0a0a0a'); fig.patch.set_facecolor('#0d1117')
ax.bar(x - w, p_vals, w, label='Precision', color='#00e5ff', alpha=0.85)
ax.bar(x,     r_vals, w, label='Recall',    color='#00ff41', alpha=0.85)
ax.bar(x + w, f_vals, w, label='F1',        color='#ffd700', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(itypes, rotation=30, ha='right', color='white', fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', color='white')
ax.set_title('P/R/F1 per Issue Type — 5% Noise (avg across 4 datasets)', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.spines[:].set_color('#333')
ax.grid(axis='y', alpha=0.2, color='#444')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.savefig('fig2_per_issue_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_per_issue_type.png')

In [ ]:
# ── Figure 3: Quality Score vs Noise Level ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor('#0a0a0a'); fig.patch.set_facecolor('#0d1117')

for ds in datasets_clean:
    d = df_metrics[df_metrics['dataset'] == ds].sort_values('noise_pct')
    if d.empty: continue
    ax.plot(d['noise_pct'], d['overall_score'],
            marker='s', linewidth=2, linestyle='--',
            color=COLOURS[ds], label=LABELS[ds])

ax.axhline(85, color='#ff4444', linestyle=':', linewidth=1.5, label='Pass threshold (85%)')
ax.set_xlabel('Noise Level (%)', color='white', fontsize=12)
ax.set_ylabel('Quality Score (%)', color='white', fontsize=12)
ax.set_title('Quality Score Degradation vs Injected Noise', color='white', fontsize=13, pad=15)
ax.set_ylim(0, 105)
ax.set_xticks(NOISE_TICKS)
ax.set_xticklabels([f'{n}%' for n in NOISE_TICKS], color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#333')
ax.grid(alpha=0.2, color='#444')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=10)
plt.tight_layout()
plt.savefig('fig3_quality_score_vs_noise.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_quality_score_vs_noise.png')

In [ ]:
# ── Figure 4: Heatmap — F1 per Dataset x Noise Level ──────────────────────
pivot = df_metrics.pivot(index='dataset', columns='noise_pct', values='overall_f1')

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor('#0d1117')
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'F1 Score'})
ax.set_title('F1 Score Heatmap — Dataset × Noise Level', color='white', fontsize=12, pad=12)
ax.set_xlabel('Noise Level (%)', color='white')
ax.set_ylabel('Dataset', color='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('fig4_f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig4_f1_heatmap.png')

## 11. Export

In [ ]:
df_metrics.drop(columns=['per_type'], errors='ignore').to_csv('evaluation_results_aizle.csv', index=False)
print('Saved: evaluation_results_aizle.csv')

if IN_COLAB:
    for f in ['evaluation_results_aizle.csv', 'fig1_f1_vs_noise.png',
              'fig2_per_issue_type.png', 'fig3_quality_score_vs_noise.png',
              'fig4_f1_heatmap.png']:
        try:
            files.download(f)
        except Exception as e:
            print(f'Could not download {f}: {e}')

print('\nDone.')